In [28]:
# NOTEBOOK 3: MODEL TESTING
# Student Performance Prediction - Testing with New Inputs

import pandas as pd
import numpy as np
import joblib
import os

print("=" * 50)
print("NOTEBOOK 3: MODEL TESTING")
print("=" * 50)

# 1. LOAD SAVED MODELS
print("\nSTEP 1: LOADING SAVED MODELS")
print("-" * 30)

# Check if models folder exists
if not os.path.exists("models"):
    print("ERROR: 'models' folder not found!")
    print("Please run Notebook 2 first to train and save the models.")
    print("Stopping execution...")
    raise FileNotFoundError("Models folder not found. Run Notebook 2 first.")

# Check if each model file exists
model_files = ["best_model.pkl", "scaler.pkl", "label_encoder.pkl"]
for file in model_files:
    if not os.path.exists(f"models/{file}"):
        print(f"ERROR: models/{file} not found!")
        print("Please run Notebook 2 first to train and save the models.")
        raise FileNotFoundError(f"models/{file} not found. Run Notebook 2 first.")

# Load models
model = joblib.load("models/best_model.pkl")
scaler = joblib.load("models/scaler.pkl")
encoder = joblib.load("models/label_encoder.pkl")

print("Models loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Classes: {encoder.classes_}")

# 2. CREATE TEST SAMPLES (Different student profiles)
print("\nSTEP 2: CREATING TEST SAMPLES")
print("-" * 30)

# Test Sample 1: Likely Graduate (High grades, good attendance)
sample_graduate = [
    1, 17, 1, 9500, 1,  # Marital, App mode, App order, Course, Day/Evening
    1, 160.0, 1, 19, 38,  # Prev qual, Prev grade, Nationality, Mother qual, Father qual
    9, 9, 150.0, 0,  # Mother occ, Father occ, Admission grade, Displaced
    0, 0, 1, 1, 1,  # Edu needs, Debtor, Tuition paid, Gender, Scholarship
    20, 0,  # Age, International
    0, 6, 6, 6, 14.0, 0,  # 1st sem: credited, enrolled, eval, approved, grade, without eval
    0, 6, 6, 6, 14.0, 0,  # 2nd sem: credited, enrolled, eval, approved, grade, without eval
    10.8, 1.4, 1.74  # Unemployment, Inflation, GDP
]

# Test Sample 2: Likely Dropout (Low grades, financial issues)
sample_dropout = [
    1, 39, 1, 9991, 0,
    1, 100.0, 1, 19, 38,
    9, 9, 100.0, 0,
    0, 1, 0, 1, 0,
    30, 0,
    0, 5, 0, 0, 0.0, 0,
    0, 5, 0, 0, 0.0, 0,
    13.9, -0.3, 0.79
]

# Test Sample 3: Average Student (Enrolled)
sample_enrolled = [
    1, 1, 2, 9238, 1,
    1, 125.0, 1, 19, 38,
    9, 9, 120.0, 0,
    0, 0, 1, 1, 0,
    22, 0,
    0, 6, 6, 5, 12.0, 0,
    0, 6, 6, 5, 11.5, 0,
    11.1, 0.6, 2.02
]

print("Test samples created:")
print("  - Sample 1: High performing student (Expected: Graduate)")
print("  - Sample 2: At-risk student (Expected: Dropout)")
print("  - Sample 3: Average student (Expected: Enrolled)")

# 3. MAKE PREDICTIONS
print("\nSTEP 3: MAKING PREDICTIONS")
print("-" * 30)

test_samples = [sample_graduate, sample_dropout, sample_enrolled]
sample_names = ["High Performing Student", "At-Risk Student", "Average Student"]
expected = ["Graduate", "Dropout", "Enrolled"]

for i, (sample, name, exp) in enumerate(zip(test_samples, sample_names, expected)):
    # Convert to numpy array and reshape
    sample_array = np.array(sample).reshape(1, -1)

    # Scale the features
    sample_scaled = scaler.transform(sample_array)

    # Predict
    prediction_encoded = model.predict(sample_scaled)
    prediction = encoder.inverse_transform(prediction_encoded)[0]

    print(f"\nSample {i+1}: {name}")
    print(f"  Expected: {exp}")
    print(f"  Predicted: {prediction}")
    print(f"  Correct: {'YES' if prediction == exp else 'NO'}")

# 4. TEST WITH RANDOM SAMPLES FROM DATASET (with error handling)
print("\nSTEP 4: TESTING WITH RANDOM SAMPLES FROM DATASET")
print("-" * 30)

# Try to load original data
try:
    # Check if data.csv exists
    if os.path.exists("data.csv"):
        df = pd.read_csv("data.csv", sep=';')
        print(f"Loaded {len(df)} records from dataset")

        # Take 3 random samples
        random_samples = df.sample(n=min(3, len(df)), random_state=42)

        correct_count = 0
        total_count = 0

        for idx, row in random_samples.iterrows():
            # Get features (all except Target)
            features = row.drop("Target").values.reshape(1, -1)
            actual = row["Target"]

            # Scale and predict
            features_scaled = scaler.transform(features)
            prediction_encoded = model.predict(features_scaled)
            prediction = encoder.inverse_transform(prediction_encoded)[0]

            is_correct = "YES" if prediction == actual else "NO"
            if is_correct == "YES":
                correct_count += 1
            total_count += 1

            print(f"\nRecord {idx}:")
            print(f"  Actual: {actual}")
            print(f"  Predicted: {prediction}")
            print(f"  Correct: {is_correct}")

        print(f"\nAccuracy on {total_count} random samples: {correct_count/total_count*100:.1f}%")

    else:
        print("data.csv not found. Skipping dataset testing.")
        print("If you want to test with real data, please upload data.csv to Colab.")

except Exception as e:
    print(f"Could not load dataset: {e}")
    print("Skipping dataset testing.")

# 5. BATCH TESTING FUNCTION
print("\nSTEP 5: BATCH TESTING FUNCTION")
print("-" * 30)

def predict_student(features_list):
    """
    Predict student outcomes for one or multiple students

    Parameters:
    features_list: List of feature arrays or single feature array

    Returns:
    List of predictions
    """
    if len(np.array(features_list).shape) == 1:
        features_list = [features_list]

    features_array = np.array(features_list)
    features_scaled = scaler.transform(features_array)
    predictions_encoded = model.predict(features_scaled)
    predictions = encoder.inverse_transform(predictions_encoded)

    return predictions

print("Function 'predict_student()' created successfully!")
print("\nExample usage:")
print("  prediction = predict_student(sample_graduate)")
print("  print(prediction)  # Output: ['Graduate']")

# 6. QUICK VERIFICATION TEST
print("\nSTEP 6: QUICK VERIFICATION TEST")
print("-" * 30)

# Test all three samples
grad_result = predict_student(sample_graduate)
drop_result = predict_student(sample_dropout)
enr_result = predict_student(sample_enrolled)

print(f"Graduate sample -> Prediction: {grad_result[0]}")
print(f"Dropout sample -> Prediction: {drop_result[0]}")
print(f"Enrolled sample -> Prediction: {enr_result[0]}")

# Final summary
print("\n" + "=" * 50)
print("MODEL TESTING COMPLETE!")
print("=" * 50)
print("\nSummary:")
print(f"  - Model loaded: {type(model).__name__}")
print(f"  - Number of classes: {len(encoder.classes_)}")
print(f"  - Classes: {encoder.classes_}")
print(f"  - Batch prediction function ready: predict_student()")
print("\nNOTE: To test with your own data, use:")
print("  result = predict_student(your_data_list)")

NOTEBOOK 3: MODEL TESTING

STEP 1: LOADING SAVED MODELS
------------------------------
Models loaded successfully!
Model type: LogisticRegression
Classes: ['Dropout' 'Enrolled' 'Graduate']

STEP 2: CREATING TEST SAMPLES
------------------------------
Test samples created:
  - Sample 1: High performing student (Expected: Graduate)
  - Sample 2: At-risk student (Expected: Dropout)
  - Sample 3: Average student (Expected: Enrolled)

STEP 3: MAKING PREDICTIONS
------------------------------

Sample 1: High Performing Student
  Expected: Graduate
  Predicted: Enrolled
  Correct: NO

Sample 2: At-Risk Student
  Expected: Dropout
  Predicted: Enrolled
  Correct: NO

Sample 3: Average Student
  Expected: Enrolled
  Predicted: Enrolled
  Correct: YES

STEP 4: TESTING WITH RANDOM SAMPLES FROM DATASET
------------------------------
Loaded 4424 records from dataset

Record 1255:
  Actual: Dropout
  Predicted: Enrolled
  Correct: NO

Record 3458:
  Actual: Graduate
  Predicted: Enrolled
  Correct: 